In [1]:
import json, pandas as pd
from pathlib import Path
R=Path('results') if Path('results').exists() else Path('experiments/E12C_gpt_prompt_confirmation/results')
d=json.load(open(R/'e12c_analysis.json')); S=d['E12C_standalone']; K=['p0','p3']
pd.set_option('display.width',220,'display.max_colwidth',90)

# E12C — GPT-P0 vs GPT-P3 confirmation (Stage B, final prompt-selection experiment)
## 1. Research question
Does GPT-P3 reproduce its directional advantage over P0 on a fresh, document-disjoint TRAIN confirmation manifest?

## 2. Why confirmation was needed
E12B: P3 122 vs P0 114 joint of 150 (+8, one short of +9), all CIs included zero, Contradiction Recall unchanged.

## 3. Fresh-manifest controls
TRAIN_GPT_PROMPT_CONFIRM_v1: 150 cases (50/50/50), 67 docs, seed 1300; document-disjoint from TRAIN_PROMPT_v1, TRAIN_ARCH_v1, TRAIN_ORACLE_v1, TRAIN_GPT_PROMPT_v1.

In [2]:
print(json.load(open(R/'manifest_inspection.json'))['inspection'])

{'total_train_docs': 423, 'total_train_cases': 7191, 'docs_TRAIN_PROMPT_v1': 73, 'docs_TRAIN_ARCH_v1': 78, 'overlap_PROMPT_ARCH_docs': 19, 'overlap_PROMPT_ORACLE_docs': 29, 'overlap_ARCH_ORACLE_docs': 29, 'union_of_four_prior_manifests_docs': 301, 'untouched_docs': 122, 'untouched_label_distribution_cases': {'NotMentioned': 860, 'Entailment': 978, 'Contradiction': 236}, 'docs_TRAIN_ORACLE_v1': 145, 'docs_TRAIN_GPT_PROMPT_v1': 74, 'overlap_GPT_with_any_earlier': 0}


## 4. Frozen setup
P0/P3 byte-identical to E12B (full sha1 verified before the run), retrieval_v1 top-5 (one shared artifact), openai/gpt-5-mini via OpenRouter, temperature 0.0, 60s timeout, concurrency 5 within each arm, arms sequential P0→P3, unchanged schema/parser/validator/scorer.

## 5. Execution integrity

In [3]:
print(json.load(open(R/'pre_run_verification.json'))['checks']); print(json.dumps(d['run'],indent=1))

{'manifest_seed_1200_150': True, 'context_same_ids_order': True, 'context_top5': True, 'context_no_gold_leakage': True, 'prompt_full_hashes_match_frozen': True, 'ledger_approx_1_7637': True, 'no_existing_e12c_ledger_rows': True, 'e12b_ledger_rows_present_600': True, 'no_existing_e12c_outputs': True, 'budget_gate_pass': True}
{
 "arms": {
  "gpt_p0": {
   "spend_usd": 0.23581474999999988,
   "wall_seconds": 201.7100523749832,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_calls": 0,
   "verify": {
    "records_150": true,
    "no_duplicates": true,
    "all_case_ids_present": true,
    "all_successful": true,
    "prompt_hash_ok": true,
    "run_id_ok": true
   }
  },
  "gpt_p3": {
   "spend_usd": 0.23223775,
   "wall_seconds": 186.6650791669963,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_calls": 0,
   "verify": {
    "records_150": true,
    "no_duplicates": true,
    "all_case_ids_present": true,
    "all_successful": true,
    "prompt_hash_ok": tru

## 6. GPT-P0 standalone metrics

In [4]:
s=S['p0']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall']); print('joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.8933, 'macro_f1': 0.8934, 'evidence_recall': 0.92, 'evidence_precision': 0.9293, 'source_valid_evidence_rate': 0.9495, 'joint': 0.8533}
recall {'Entailment': 0.92, 'Contradiction': 0.82, 'NotMentioned': 0.94}
joint_by_class {'Entailment': 0.86, 'Contradiction': 0.76, 'NotMentioned': 0.94}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[46, 2, 2], [7, 41, 2], [3, 0, 47]]


## 7. GPT-P3 standalone metrics

In [5]:
s=S['p3']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall']); print('joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.8667, 'macro_f1': 0.8671, 'evidence_recall': 0.9, 'evidence_precision': 0.8911, 'source_valid_evidence_rate': 0.9505, 'joint': 0.8333}
recall {'Entailment': 0.86, 'Contradiction': 0.84, 'NotMentioned': 0.9}
joint_by_class {'Entailment': 0.8, 'Contradiction': 0.8, 'NotMentioned': 0.9}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[43, 4, 3], [7, 42, 1], [3, 2, 45]]


## 8. Matched comparison

In [6]:
rows={'Accuracy':'accuracy','Macro-F1':'macro_f1','Evidence recall':'evidence_recall','Evidence precision':'evidence_precision','Source-valid evidence':'source_valid_evidence_rate','Joint':'joint'}
t=pd.DataFrame({k:{n:S[k][m] for n,m in rows.items()} for k in K})
for c in ['Entailment','Contradiction','NotMentioned']: t.loc[c+' recall']=[S[k]['recall'][c] for k in K]
for c in ['Entailment','Contradiction','NotMentioned']: t.loc['Joint '+c]=[S[k]['joint_by_class'][c] for k in K]
t.loc['Joint (cases/150)']=[S[k]['joint_n'] for k in K]; t['delta']=t.p3-t.p0; t.round(4)

,p0,p3,delta
Accuracy,0.8933,0.8667,-0.0267
Macro-F1,0.8934,0.8671,-0.0264
Evidence recall,0.9200,0.9000,-0.0200
Evidence precision,0.9293,0.8911,-0.0382
Source-valid evidence,0.9495,0.9505,0.0010
Joint,0.8533,0.8333,-0.0200
Entailment recall,0.9200,0.8600,-0.0600
Contradiction recall,0.8200,0.8400,0.0200
NotMentioned recall,0.9400,0.9000,-0.0400
Joint Entailment,0.8600,0.8000,-0.0600


## 9-10. Classification and joint transitions (discordant IDs)

In [7]:
p=d['paired_E12C']; print('CLS',p['cls']); print('JOINT',p['joint'])
for k in ['cls_wrong_to_right_ids','cls_right_to_wrong_ids','joint_fail_to_success_ids','joint_success_to_fail_ids']: print(k,p[k])

CLS {'wrong_to_right': 1, 'right_to_wrong': 5, 'both_right': 129, 'both_wrong': 15}
JOINT {'wrong_to_right': 2, 'right_to_wrong': 5, 'both_right': 123, 'both_wrong': 20}
cls_wrong_to_right_ids ['train::269::nda-20']
cls_right_to_wrong_ids ['train::215::nda-3', 'train::481::nda-10', 'train::620::nda-10', 'train::112::nda-1', 'train::583::nda-1']
joint_fail_to_success_ids ['train::148::nda-1', 'train::269::nda-20']
joint_success_to_fail_ids ['train::215::nda-3', 'train::347::nda-12', 'train::620::nda-10', 'train::112::nda-1', 'train::583::nda-1']


## 11. Contradiction analysis

In [8]:
print(pd.DataFrame({k:d['contradiction'][k] for k in K})); print('recoveries',d['contradiction']['recoveries'],'regressions',d['contradiction']['regressions'])

                    p0     p3
correct          41.00  42.00
of               50.00  50.00
recall            0.82   0.84
to_Entailment     7.00   7.00
to_NotMentioned   2.00   1.00
recoveries ['train::269::nda-20'] regressions []


## 12. Evidence analysis (note: 'wrong_label_source_valid' excludes wrong-label cases whose evidence also overlaps gold, which are counted in the gold-overlap row; '(…incl_overlap)' counts overlap∩source-valid)

In [9]:
pd.DataFrame(d['evidence_categories']).fillna(0).astype(int)

,p0,p3
correct_label_gold_valid_evidence,128,125
wrong_label_gold_overlap_evidence,7,7
(wrong_label_source_valid_evidence_incl_overlap),6,7
correct_label_bad_evidence,6,5
wrong_label_other,5,6
wrong_label_source_valid_evidence,4,7


## 13. E12C paired statistics (supporting evidence)

In [10]:
print('McNemar cls',p['mcnemar_cls']); print('McNemar joint',p['mcnemar_joint']); print({a:(round(b['point'],3),[round(x,3) for x in b['ci95']]) for a,b in p['bootstrap'].items()})

McNemar cls {'p0_only': 5, 'candidate_only': 1, 'n_discordant': 6, 'p': 0.21875}
McNemar joint {'p0_only': 5, 'candidate_only': 2, 'n_discordant': 7, 'p': 0.453125}
{'accuracy': (-0.027, [-0.06, 0.0]), 'macro_f1': (-0.026, [-0.06, 0.003]), 'contradiction_recall': (0.02, [0.0, 0.067]), 'joint': (-0.02, [-0.053, 0.013])}


## 14. E12B recap

In [11]:
print(d['E12B_recap']); print(d['E12B_recap_summary'])

{'p0_joint': 114, 'p3_joint': 122, 'delta': 8, 'of': 150}
{'p0': {'accuracy': 0.8133333333333334, 'macro_f1': 0.8136195909652019, 'joint_n': 114, 'evidence_recall': 0.85, 'evidence_precision': 0.8252427184466019}, 'p3': {'accuracy': 0.84, 'macro_f1': 0.8402777777777778, 'joint_n': 122, 'evidence_recall': 0.92, 'evidence_precision': 0.8846153846153846}}


## 15. Pooled E12B + E12C (manifest identity kept explicit)

In [12]:
print('E12B',d['E12B_recap']); print('E12C',d['E12C']); print('POOLED joint P0/P3 of 300:',d['pooled']['joint_p0_of_300'],d['pooled']['joint_p3_of_300'],'net',d['pooled']['net']); pd.DataFrame(d['pooled']['stats'])

E12B {'p0_joint': 114, 'p3_joint': 122, 'delta': 8, 'of': 150}
E12C {'p0_joint': 128, 'p3_joint': 125, 'delta': -3, 'of': 150}
POOLED joint P0/P3 of 300: 242 247 net 5


,p0,p3
n,300.000000,300.000000
joint,242.000000,247.000000
accuracy,0.853333,0.853333
macro_f1,0.853688,0.853863
contradiction_correct,82.000000,83.000000
contradiction_n,100.000000,100.000000
contradiction_recall,0.820000,0.830000
evidence_recall,0.885000,0.910000
evidence_precision,0.876238,0.887805


## 16. Stratified pooled bootstrap (10,000 draws, resampling within each manifest)

In [13]:
print(json.dumps(d['pooled']['stratified_bootstrap'],indent=1)); print('per-manifest joint bootstrap',d['pooled']['per_manifest_bootstrap_joint'])

{
 "joint": {
  "point": 0.01666666666666672,
  "ci95": [
   -0.01666666666666672,
   0.050000000000000044
  ]
 },
 "accuracy": {
  "point": 0.0,
  "ci95": [
   -0.033333333333333326,
   0.033333333333333326
  ]
 },
 "macro_f1": {
  "point": 0.00017450034621335764,
  "ci95": [
   -0.032664122877606006,
   0.03283731126155209
  ]
 },
 "contradiction_recall": {
  "point": 0.010000000000000009,
  "ci95": [
   -0.032258064516129004,
   0.05555555555555558
  ]
 }
}
per-manifest joint bootstrap {'E12B': {'point': 0.053333333333333344, 'ci95': [-0.00666666666666671, 0.1133333333333334]}, 'E12C': {'point': -0.020000000000000018, 'ci95': [-0.053333333333333344, 0.013333333333333308]}}


## 17. Manifest-direction consistency
**The direction reversed**: E12B +8, E12C −3. The pooled +5 must not be read as replication.

In [14]:
d['direction_consistency']

{'E12B_delta': 8,
 'E12C_delta': -3,
 'same_direction': False,
 'reversal': True,
 'note': 'If E12C reverses vs E12B this is stated explicitly regardless of the pooled aggregate.'}

## 18. Cost / latency (per-call latency; not arm wall-clock)

In [15]:
pd.DataFrame({k:{**{x:v for x,v in d['cost_latency'][k].items() if x!='latency_ms'},**{'lat_'+x:v for x,v in d['cost_latency'][k]['latency_ms'].items()}} for k in K}); 

In [16]:
d['cost_latency']['delta']

{'cost_pct': -1.5168686437129097,
 'latency_mean_pct': -5.27130514875428,
 'input_tokens': 130.0,
 'output_tokens': -28.173333333333403}

## 19. Guard evaluation (frozen)

In [17]:
pd.DataFrame(d['guards']).T

,delta_cases,pass,net_joint_cases,delta
1_contradiction_not_3_or_more_cases_below_P0,1,True,NaN,NaN
2_net_joint_positive,NaN,False,-3,NaN
3_macro_f1_drop_not_over_0.03,NaN,True,NaN,-0.026384
4_evidence_recall_drop_not_over_3pp,NaN,True,NaN,-0.02
5_evidence_precision_drop_not_over_3pp,NaN,False,NaN,-0.038204


## 20. Confirmation band

In [18]:
print(d['confirmation_band'])

NO CONFIRMATION


## 21. Final A/B/C decision

In [19]:
print(d['outcome'])

B - P3 NOT CONFIRMED - KEEP P0


## 22. Prompt-freeze action
Outcome B → no `classification_prompt_gpt_v1` created; P0 (`classification_prompt_v1`) is kept. GPT prompt optimisation is closed.

In [20]:
from pathlib import Path; print('classification_prompt_gpt_v1 exists:', any(Path('../../prompts/final').glob('classification_prompt_gpt_v1*')) if Path('../../prompts/final').exists() else any(Path('prompts/final').glob('classification_prompt_gpt_v1*')))

classification_prompt_gpt_v1 exists: False
